In [ ]:
# CatBoost

https://catboost.ai/

CatBoost는 Yandex에서 개발한 Gradient Boosting 기반 알고리즘으로, 범주형 데이터 처리에 강점을 가진 머신러닝 모델이다.

이미 boosting의 기본 철학과, 실무 라이브러리가 그 철학을 어떻게 빠르고 강하게 구현하는지 보았다.
이제 CatBoost에서는 같은 boosting 계열 안에서도 범주형 데이터를 얼마나 자연스럽게 다룰 수 있는가라는 문제를 본다.

1. 좋은 boosting 모델은 성능만이 아니라 데이터 형태에 얼마나 잘 맞는지도 중요하다.
2. CatBoost는 범주형 데이터를 다룰 때 전처리 부담을 줄이도록 설계된 boosting 라이브러리이다.
3. 따라서 점수만 보는 것이 아니라, 범주형 열 지정 방식, Pool 객체, 특성 중요도까지 함께 봐야 CatBoost의 강점이 보인다.

이름 유래:
"CatBoost" = "Categorical + Boosting"

1. 범주형 데이터 처리에 강점
   * 원-핫 인코딩 없이도 범주형 데이터를 효과적으로 다룰 수 있다.
   * 범주형 값을 처리할 때 순서를 활용한 통계 기반 인코딩 방식을 사용하여, 타깃 누수를 줄이도록 설계되었다.
   * 이때 현재 샘플의 정답을 미리 사용하지 않도록 순서를 고려하여 인코딩함으로써 타깃 누수를 줄이도록 설계되었다.

2. 안정적인 학습 구조
   * 데이터를 보는 순서에 따라 인코딩 결과가 과하게 치우치지 않도록 permutation 기반 아이디어를 활용한다.
   * 이를 통해 데이터 순서에 대한 민감도를 줄이고, 보다 안정적으로 학습할 수 있다.
   * 예를 들어 같은 범주형 값이라도 현재 행의 정답까지 미리 반영해버리면 누수가 생길 수 있는데, CatBoost는 이런 문제를 줄이도록 설계되어 있다.

3. 빠른 학습 및 예측
   * 대규모 데이터셋에서도 효율적으로 동작하도록 설계되었다.
   * 특히 범주형 데이터 전처리를 따로 많이 하지 않아도 되어, 실무에서 전체 파이프라인을 단순하게 구성하기 좋다.

## 환경설정

In [ ]:
# %conda install catboost

## 간단예제 구현

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = {
    'gender': ['Male', 'Female', 'Female', 'Male', 'Female'],
    'region': ['North', 'South', 'East', 'West', 'North'],
    'membership_type': ['Basic', 'Premium', 'Basic', 'Basic', 'Premium'],
    'age': [23, 35, 45, 50, 27],
    'purchased': [0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)
df

,gender,region,membership_type,age,purchased
0,Male,North,Basic,23,0
1,Female,South,Premium,35,1
2,Female,East,Basic,45,0
3,Male,West,Basic,50,0
4,Female,North,Premium,27,1


In [3]:
# 데이터 전처리
from sklearn.model_selection import train_test_split

X = df.drop('purchased', axis=1)
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from catboost import Pool, CatBoostClassifier
from sklearn.metrics import accuracy_score
# 범주형 데이터 정의
cat_features = ['gender','region','membership_type']

# Pool : CatBoost 전용 데이터 객체로 입력 데이터, 타깃, 범주형 열 정보 등을 함께 묶어 전달한다.
train_pool = Pool(X_train, y_train, cat_features = cat_features)
test_pool = Pool(X_test, y_test, cat_features = cat_features)

# 모델 학습
cat_clf = CatBoostClassifier(
    iterations=100,      # boosting round 수
    depth=3,             # 트리 깊이
    learning_rate=0.1,   # 학습률
    verbose=0
)

# Pool 객체를 전달하며 학습
cat_clf.fit(train_pool)

print('accuracy:', accuracy_score(y_test, cat_clf.predict(test_pool)))

accuracy: 1.0


In [5]:
# 특성 중요도
simple_importance_df = pd.DataFrame({
    'feature' : X_train.columns,
    'importance' : cat_clf.get_feature_importance(train_pool)
}).sort_values('importance', ascending=False)

simple_importance_df

,feature,importance
2,membership_type,83.249902
3,age,7.961470
0,gender,7.901964
1,region,0.886663


In [6]:
# 각 클래스에 속할 확률
print(cat_clf.predict_proba(test_pool))

[[0.28663273 0.71336727]]


## Adult Income


In [7]:
# 데이터로드
data_df = pd.read_csv('data/adult_income.csv')
data_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [8]:
# 데이터 분할
X = data_df.drop('income', axis=1)
y = data_df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
# 일반 모델을 사용한다면 범주형에 대한 전처리 필요 (Logis)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
                        'relationship','race','sex','native-contry']

numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer({
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features)
})

baseline_model = Pipeline({
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=10000))
})

baseline_model.fit(X_train, y_train)

TypeError: unhashable type: 'list'

In [ ]:
# 모델 학습
categorical_features=[""]